# ДЗ №2. Задание 2
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Расстояния поездок.


## Подготовка


In [2]:
from pyspark import SparkConf
from pyspark.sql import SparkSession, functions as F

conf = SparkConf().setMaster("local[2]").setAppName("HW2-Distances")
spark = SparkSession.builder.config(conf=conf).getOrCreate()

trips = spark.read.csv("../*citibike-tripdata*.csv", header=True)

## 1. Расстояния поездок

Исключаем поездки с одинаковыми ID начальной и конечной станций. Строки без ID станции тоже не участвуют.
Переводим координаты в радианы и считаем расстояние по формуле гаверсинуса

In [5]:
coordinates = (
    trips.filter("start_station_id != end_station_id")
    .selectExpr(
        "radians(cast(start_lat as double)) as lat1",
        "radians(cast(start_lng as double)) as lon1",
        "radians(cast(end_lat as double)) as lat2",
        "radians(cast(end_lng as double)) as lon2"
    )
)

lat1, lon1 = F.col("lat1"), F.col("lon1")
lat2, lon2 = F.col("lat2"), F.col("lon2")
a = (
    F.sin((lat2 - lat1) / 2) ** 2
    + F.cos(lat1) * F.cos(lat2) * F.sin((lon2 - lon1) / 2) ** 2
)
distances = coordinates.select(
    (2 * 6371000 * F.asin(F.sqrt(a))).alias("distance_m")
)

## 2. Статистика расстояний

Максимум, среднее, стандартное отклонение и медиана в метрах

In [6]:
statistics = distances.agg(
    F.round(F.max("distance_m"), 2).alias("Максимум, м"),
    F.round(F.avg("distance_m"), 2).alias("Среднее, м"),
    F.round(F.stddev_pop("distance_m"), 2).alias("Станд. отклонение, м"),
    F.round(F.median("distance_m"), 2).alias("Медиана, м")
)
statistics.show()

+-----------+----------+--------------------+----------+
|Максимум, м|Среднее, м|Станд. отклонение, м|Медиана, м|
+-----------+----------+--------------------+----------+
|   27103.73|   2045.47|             1685.39|    1538.0|
+-----------+----------+--------------------+----------+



26/11/17 17:24:37 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 262016 ms exceeds timeout 120000 ms
26/11/17 17:24:37 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/17 17:24:44 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$